In [ ]:
"""
Real-time multi-camera pose streaming.

Cameras are calibrated once from the first frame pair.
Every subsequent frame: detect → triangulate → solve → yield.
"""

import numpy as np
import torch
import cv2
import math
import roma
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from xfeat_c.xfeat import XFeat
from scipy.spatial.transform import Rotation as Rot
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix

import sys
sys.path.insert(0, "/Users/saptarshiMT/Music/meshsense/claude_code/anny/src")
import anny

# =============================================================================
# CONFIG
# =============================================================================
DEVICE = "cpu"
DTYPE = torch.float32

VIDEOS = [
    "/Users/saptarshiMT/Music/meshsense/camera0.avi",
    "/Users/saptarshiMT/Music/meshsense/camera1.avi",
]
WIDTH, HEIGHT = 640, 480
MEDIAPIPE_MODEL = "/Users/saptarshiMT/Downloads/pose_landmarker_heavy.task"

PHENOTYPE_KWARGS = {
    "gender": 1, "age": 0.5, "height": 0.6, "weight": 0.1, "muscle": 0.1,
}

M = np.array([[1, 0, 0],
              [0, 0, 1],
              [0, -1, 0]], dtype=np.float64)

MEDIAPIPE_TO_ANNY = {
    0: "head", 1: "eye.L", 2: "eye.L", 3: "eye.L",
    4: "eye.R", 5: "eye.R", 6: "eye.R",
    7: "head", 8: "head",
    9: "oris04.L", 10: "oris04.R",
    11: "shoulder01.L", 12: "shoulder01.R",
    13: "lowerarm01.L", 14: "lowerarm01.R",
    15: "wrist.L", 16: "wrist.R",
    17: "finger5-1.L", 18: "finger5-1.R",
    19: "finger2-1.L", 20: "finger2-1.R",
    21: "finger1-1.L", 22: "finger1-1.R",
    23: "pelvis.L", 24: "pelvis.R",
    25: "lowerleg01.L", 26: "lowerleg01.R",
    27: "foot.L", 28: "foot.R",
    29: "foot.L", 30: "foot.R",
    31: "toe2-1.L", 32: "toe2-1.R",
}


# =============================================================================
# CAMERA INTRINSICS
# =============================================================================

def get_camera_matrix(w, h):
    sx, sy = w / 1920, h / 1080
    return np.array([
        [1950.78 * sx, 0.0,          960.0 * sx],
        [0.0,          1950.78 * sy, 540.0 * sy],
        [0.0,          0.0,          1.0       ],
    ], dtype=np.float64)


# =============================================================================
# MEDIAPIPE
# =============================================================================

def create_pose_landmarker(model_path):
    opts = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=model_path),
        running_mode=vision.RunningMode.IMAGE,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.PoseLandmarker.create_from_options(opts)


def get_pose_points(image, landmarker):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not result.pose_landmarks:
        return {}, {}
    h, w = image.shape[:2]
    pts, vis = {}, {}
    for idx, lm in enumerate(result.pose_landmarks[0]):
        x, y = int(lm.x * w), int(lm.y * h)
        if 0 <= x < w and 0 <= y < h:
            pts[idx] = (x, y)
            vis[idx] = float(lm.visibility) if hasattr(lm, "visibility") else 1.0
    return pts, vis


# =============================================================================
# CAMERA POSE ESTIMATION (run once)
# =============================================================================

def estimate_cameras(frames, K):
    """Estimate all camera poses relative to camera 0. Run once."""
    xfeat = XFeat()
    Rs, ts, Ps = [np.eye(3)], [np.zeros(3)], [K @ np.hstack([np.eye(3), np.zeros((3, 1))])]

    for i in range(1, len(frames)):
        kp0, kpi = xfeat.match_xfeat_star(frames[0], frames[i], top_k=10000)
        pts0 = np.asarray(kp0, dtype=np.float64)
        ptsi = np.asarray(kpi, dtype=np.float64)
        E, mask = cv2.findEssentialMat(pts0, ptsi, K, method=cv2.RANSAC, prob=0.999, threshold=2.0)
        _, R, t, _ = cv2.recoverPose(E, pts0, ptsi, K, mask=mask)
        Rs.append(R)
        ts.append(t.flatten())
        Ps.append(K @ np.hstack([R, t]))

    return Rs, ts, Ps


# =============================================================================
# TRIANGULATION (DLT)
# =============================================================================

def dlt_triangulate_point(Ps, pts):
    A = []
    for P, (x, y) in zip(Ps, pts):
        A.append(x * P[2] - P[0])
        A.append(y * P[2] - P[1])
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    Xh = Vt[-1]
    return Xh[:3] / Xh[3]


def triangulate_landmarks(Ps, pts2d_list):
    all_ids = set()
    for pts in pts2d_list:
        all_ids |= set(pts.keys())
    points3d = {}
    for idx in all_ids:
        valid_Ps  = [Ps[ci] for ci, pts in enumerate(pts2d_list) if idx in pts]
        valid_pts = [pts[idx] for pts in pts2d_list if idx in pts]
        if len(valid_Ps) >= 2:
            points3d[idx] = dlt_triangulate_point(valid_Ps, valid_pts)
    return points3d


# =============================================================================
# BUNDLE ADJUSTMENT (optional, per-frame)
# =============================================================================

def _project(K, R, t, X):
    p = K @ (R @ X + t)
    return p[:2] / p[2]


def bundle_adjustment(K, Rs, ts, points3d, pts2d_list, lm_ids, ftol=1e-5, max_nfev=200):
    N, M = len(Rs), len(lm_ids)
    lm_to_idx = {lm: i for i, lm in enumerate(lm_ids)}

    cam_indices, pt_indices = [], []
    for ci, pts2d in enumerate(pts2d_list):
        for lm in lm_ids:
            if lm in pts2d:
                cam_indices.append(ci)
                pt_indices.append(lm_to_idx[lm])
    cam_indices = np.array(cam_indices)
    pt_indices = np.array(pt_indices)

    # pack
    x0_cams = []
    for i in range(1, N):
        rvec, _ = cv2.Rodrigues(Rs[i])
        x0_cams.append(np.concatenate([rvec.flatten(), ts[i]]))
    x0_cams = np.concatenate(x0_cams) if x0_cams else np.array([])
    x0_pts = np.concatenate([points3d[lm] for lm in lm_ids])
    x0 = np.concatenate([x0_cams, x0_pts])

    def unpack(x):
        _Rs, _ts = [Rs[0]], [ts[0]]
        for i in range(N - 1):
            R, _ = cv2.Rodrigues(x[i*6:i*6+3])
            _Rs.append(R)
            _ts.append(x[i*6+3:i*6+6])
        pts = x[(N-1)*6:].reshape(M, 3)
        return _Rs, _ts, pts

    def residuals(x):
        _Rs, _ts, pts = unpack(x)
        res = []
        for ci, pi in zip(cam_indices, pt_indices):
            obs = pts2d_list[ci][lm_ids[pi]]
            prj = _project(K, _Rs[ci], _ts[ci], pts[pi])
            res.extend([prj[0] - obs[0], prj[1] - obs[1]])
        return np.array(res)

    # sparsity
    n_params = (N - 1) * 6 + M * 3
    n_res = len(cam_indices) * 2
    J = lil_matrix((n_res, n_params), dtype=np.int8)
    for k, (ci, pi) in enumerate(zip(cam_indices, pt_indices)):
        row = k * 2
        if ci > 0:
            col = (ci - 1) * 6
            J[row:row+2, col:col+6] = 1
        col_pt = (N - 1) * 6 + pi * 3
        J[row:row+2, col_pt:col_pt+3] = 1

    result = least_squares(residuals, x0, jac_sparsity=J.tocsr(), method="trf",
                           loss="huber", f_scale=1.0, ftol=ftol, xtol=ftol,
                           max_nfev=max_nfev, verbose=0)

    _Rs, _ts, pts = unpack(result.x)
    _Ps = [K @ np.hstack([R, t.reshape(3, 1)]) for R, t in zip(_Rs, _ts)]
    return _Rs, _ts, _Ps, {lm: pts[i] for i, lm in enumerate(lm_ids)}


# =============================================================================
# ANNY HELPERS  (unchanged from your code)
# =============================================================================

def build_model():
    model = anny.create_fullbody_model(
        all_phenotypes=True, local_changes=True,
        remove_unattached_vertices=True, triangulate_faces=True,
    ).to(device=DEVICE, dtype=DTYPE)
    return model


def get_parent_indices(model):
    for attr in ("bone_parents", "parent_indices", "parents", "kintree_table", "parent"):
        if hasattr(model, attr):
            val = getattr(model, attr)
            if isinstance(val, torch.Tensor):
                val = val.cpu().numpy()
            arr = np.asarray(val).flatten()
            if arr.dtype.kind in ("i", "u"):
                return arr.astype(int)
    raise AttributeError("Could not find bone parent indices")


def run_rest_pose(model):
    with torch.no_grad():
        out = model(pose_parameters={}, phenotype_kwargs=PHENOTYPE_KWARGS)
    return out, out["bone_poses"][0]


def rotation_between_vectors(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    v = np.cross(a, b)
    s, c = np.linalg.norm(v), float(np.dot(a, b))
    if s < 1e-8:
        if c > 0:
            return np.eye(3)
        perp = np.array([1., 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1., 0])
        axis = np.cross(a, perp); axis /= np.linalg.norm(axis)
        return Rot.from_rotvec(np.pi * axis).as_matrix()
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + K + K @ K * ((1 - c) / (s ** 2))


def signed_angle_about_axis(v_from, v_to, axis):
    axis = axis / np.linalg.norm(axis)
    a = v_from - np.dot(v_from, axis) * axis
    b = v_to - np.dot(v_to, axis) * axis
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return 0.0
    a /= na; b /= nb
    ang = np.arccos(np.clip(np.dot(a, b), -1.0, 1.0))
    if np.dot(np.cross(a, b), axis) < 0:
        ang = -ang
    return ang


def run_native(extra, model, pose_no_root):
    p = {**pose_no_root, **extra}
    with torch.no_grad():
        return model(pose_parameters=p, phenotype_kwargs=PHENOTYPE_KWARGS)["bone_poses"][0].cpu().numpy()


def split_about_axis(R_world, weights, limits_deg):
    rv = Rot.from_matrix(R_world).as_rotvec()
    theta = np.linalg.norm(rv)
    if theta < 1e-8:
        return [np.eye(3) for _ in weights]
    axis = rv / theta
    angles, used = [], 0.0
    for w, lim in zip(weights[:-1], limits_deg[:-1]):
        ti = float(np.clip(w * theta, -np.radians(lim), np.radians(lim)))
        angles.append(ti); used += ti
    angles.append(theta - used)
    return [Rot.from_rotvec(t * axis).as_matrix() for t in angles]


def set_param(bone, R_share, arm_params, label_to_idx, model, pose_no_root):
    bp = run_native(arm_params, model, pose_no_root)
    C = bp[label_to_idx[bone], :3, :3]
    P = C.T @ R_share @ C
    arm_params[bone] = roma.Rigid(
        linear=torch.from_numpy(np.ascontiguousarray(P)).to(device=model.device, dtype=model.dtype).unsqueeze(0),
        translation=None)


# =============================================================================
# ARM / LEG SPECS  (built fresh each frame since targets change)
# =============================================================================

def make_seg(targets_world, a, b):
    if a not in targets_world or b not in targets_world:
        return None
    v = M @ (targets_world[b] - targets_world[a])
    return v / np.linalg.norm(v)


def make_arm_spec(targets_world):
    seg = lambda a, b: make_seg(targets_world, a, b)
    return {
        "L": [
            dict(base="upperarm01.L", tip="lowerarm01.L", target=seg(11, 13),
                 chain=["clavicle.L", "shoulder01.L", "upperarm01.L"],
                 weights=[0.15, 0.25, 0.60], limits=[20, 40, 180]),
            dict(base="lowerarm01.L", tip="wrist.L", target=seg(13, 15),
                 chain=["lowerarm01.L"], weights=[1.0], limits=[180]),
            dict(base="wrist.L", tip="finger5-1.L", target=seg(15, 17),
                 chain=["wrist.L"], weights=[1.0], limits=[180]),
            dict(base="wrist.L", tip="finger2-1.L", target=seg(15, 19),
                 chain=["wrist.L"], weights=[1.0], limits=[180]),
            dict(base="wrist.L", tip="finger1-1.L", target=seg(15, 21),
                 chain=["wrist.L"], weights=[1.0], limits=[180]),
        ],
        "R": [
            dict(base="upperarm01.R", tip="lowerarm01.R", target=seg(12, 14),
                 chain=["clavicle.R", "shoulder01.R", "upperarm01.R"],
                 weights=[0.15, 0.25, 0.60], limits=[20, 40, 180]),
            dict(base="lowerarm01.R", tip="wrist.R", target=seg(14, 16),
                 chain=["lowerarm01.R"], weights=[1.0], limits=[180]),
            dict(base="wrist.R", tip="finger5-1.R", target=seg(16, 18),
                 chain=["wrist.R"], weights=[1.0], limits=[180]),
            dict(base="wrist.R", tip="finger2-1.R", target=seg(16, 20),
                 chain=["wrist.R"], weights=[1.0], limits=[180]),
            dict(base="wrist.R", tip="finger1-1.R", target=seg(16, 22),
                 chain=["wrist.R"], weights=[1.0], limits=[180]),
        ],
    }


def make_leg_spec(targets_world):
    seg = lambda a, b: make_seg(targets_world, a, b)
    return {
        "L": [
            dict(base="upperleg01.L", tip="lowerleg01.L", target=seg(23, 25),
                 chain=["upperleg01.L"], weights=[1.0], limits=[180]),
            dict(base="lowerleg01.L", tip="foot.L", target=seg(25, 27),
                 chain=["lowerleg01.L"], weights=[1.0], limits=[180]),
            dict(base="foot.L", tip="toe2-1.L", target=seg(27, 31),
                 chain=["foot.L"], weights=[1.0], limits=[180]),
        ],
        "R": [
            dict(base="upperleg01.R", tip="lowerleg01.R", target=seg(24, 26),
                 chain=["upperleg01.R"], weights=[1.0], limits=[180]),
            dict(base="lowerleg01.R", tip="foot.R", target=seg(26, 28),
                 chain=["lowerleg01.R"], weights=[1.0], limits=[180]),
            dict(base="foot.R", tip="toe2-1.R", target=seg(28, 32),
                 chain=["foot.R"], weights=[1.0], limits=[180]),
        ],
    }


# =============================================================================
# SOLVE POSE (one frame)
# =============================================================================

def solve_pose(targets_world, rest_out, rest_bone_poses, label_to_idx, model):
    # --- pelvis yaw ---
    mp_pelvis_dir = targets_world[23] - targets_world[24]
    mp_horiz = np.array([mp_pelvis_dir[0], 0, mp_pelvis_dir[2]])
    mp_horiz /= np.linalg.norm(mp_horiz)
    yaw_angle = -np.arctan2(mp_horiz[2], mp_horiz[0])
    R_anny = Rot.from_rotvec(yaw_angle * np.array([0, 0, 1])).as_matrix()
    pelvis_rot = torch.tensor([R_anny], device=DEVICE, dtype=DTYPE)

    root_rot = roma.rotvec_to_rotmat(
        torch.tensor([[-math.pi / 2.5, 0.0, 0.0]], device=DEVICE, dtype=DTYPE))

    # --- spine ---
    mp_hip_mid = 0.5 * (targets_world[23] + targets_world[24])
    mp_sh_mid  = 0.5 * (targets_world[11] + targets_world[12])
    target_spine = M @ (mp_sh_mid - mp_hip_mid)
    target_spine /= np.linalg.norm(target_spine)
    target_sh_line = M @ (targets_world[11] - targets_world[12])
    target_sh_line /= np.linalg.norm(target_sh_line)

    heads = rest_out["rest_bone_heads"][0].cpu().numpy()
    anny_base   = heads[label_to_idx["spine05"]]
    anny_sh_mid = 0.5 * (heads[label_to_idx["shoulder01.L"]] + heads[label_to_idx["shoulder01.R"]])
    cur_spine = anny_sh_mid - anny_base
    cur_spine /= np.linalg.norm(cur_spine)
    cur_sh_line = heads[label_to_idx["shoulder01.L"]] - heads[label_to_idx["shoulder01.R"]]
    cur_sh_line /= np.linalg.norm(cur_sh_line)

    R_bend = rotation_between_vectors(cur_spine, target_spine)
    sl_after_bend = R_bend @ cur_sh_line
    twist_angle = signed_angle_about_axis(sl_after_bend, target_sh_line, target_spine)
    R_twist = Rot.from_rotvec(twist_angle * target_spine).as_matrix()
    R_torso = R_twist @ R_bend
    R_step = Rot.from_rotvec(Rot.from_matrix(R_torso).as_rotvec() / 5).as_matrix()

    spine_chain = ["spine05", "spine04", "spine03", "spine02", "spine01"]
    spine_params = {}
    for name in spine_chain:
        Bi = rest_bone_poses[label_to_idx[name], :3, :3].cpu().numpy().astype(np.float64)
        Pi = Bi.T @ R_step @ Bi
        spine_params[name] = roma.Rigid(
            linear=torch.from_numpy(np.ascontiguousarray(Pi)).to(device=DEVICE, dtype=DTYPE).unsqueeze(0),
            translation=None)

    pose_no_root = {
        "pelvis.L": roma.Rigid(linear=pelvis_rot, translation=None),
        "pelvis.R": roma.Rigid(linear=pelvis_rot, translation=None),
        **spine_params,
    }

    # --- arms ---
    arm_params = {}
    arm_spec = make_arm_spec(targets_world)
    for side in ("L", "R"):
        for joint in arm_spec[side]:
            if joint["target"] is None:
                continue
            bp   = run_native(arm_params, model, pose_no_root)
            base = bp[label_to_idx[joint["base"]], :3, 3]
            tip  = bp[label_to_idx[joint["tip"]],  :3, 3]
            cur  = (tip - base); cur /= np.linalg.norm(cur)
            R_world = rotation_between_vectors(cur, joint["target"])
            shares = split_about_axis(R_world, joint["weights"], joint["limits"])
            for bone, R_share in zip(joint["chain"], shares):
                set_param(bone, R_share, arm_params, label_to_idx, model, pose_no_root)

    # --- legs ---
    leg_params = {}
    leg_spec = make_leg_spec(targets_world)
    for side in ("L", "R"):
        for joint in leg_spec[side]:
            if joint["target"] is None:
                continue
            bp   = run_native(leg_params, model, pose_no_root)
            base = bp[label_to_idx[joint["base"]], :3, 3]
            tip  = bp[label_to_idx[joint["tip"]],  :3, 3]
            cur  = (tip - base); cur /= np.linalg.norm(cur)
            R_world = rotation_between_vectors(cur, joint["target"])
            shares = split_about_axis(R_world, joint["weights"], joint["limits"])
            for bone, R_share in zip(joint["chain"], shares):
                set_param(bone, R_share, leg_params, label_to_idx, model, pose_no_root)

    pose_params = {
        "root": roma.Rigid(linear=root_rot, translation=None),
        "pelvis.L": roma.Rigid(linear=pelvis_rot, translation=None),
        "pelvis.R": roma.Rigid(linear=pelvis_rot, translation=None),
        **spine_params, **arm_params, **leg_params,
    }
    return pose_params


# =============================================================================
# STREAMING GENERATOR
# =============================================================================

def stream_pose_params(video_paths, K, model, rest_out, rest_bone_poses,
                       label_to_idx, mediapipe_model_path,
                       size=(640, 480), run_ba=False):
    """
    Generator that yields (frame_number, pose_params) for every frame.

    Camera poses are estimated ONCE from the first frame.
    Then each subsequent frame only runs: MediaPipe → triangulate → solve.

    Parameters
    ----------
    run_ba : bool
        If True, run bundle adjustment per frame (slower but more accurate).
        Default False for real-time speed.

    Yields
    ------
    (frame_idx, pose_params) or (frame_idx, None) if detection fails
    """
    n_cams = len(video_paths)

    # open all video captures
    caps = [cv2.VideoCapture(v) for v in video_paths]
    if not all(c.isOpened() for c in caps):
        raise RuntimeError("Could not open one or more video files")

    landmarker = create_pose_landmarker(mediapipe_model_path)

    # --- calibrate cameras from the first frame ---
    first_frames = []
    for cap in caps:
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError("Could not read first frame")
        first_frames.append(cv2.resize(frame, size))

    print("[calibrate] estimating camera poses from first frame ...")
    Rs, ts, Ps = estimate_cameras(first_frames, K)
    print("[calibrate] done. streaming frames ...")

    # process first frame too
    frame_idx = 0
    frames = first_frames

    try:
        while True:
            # detect landmarks in all cameras
            pts2d_list, vis_list = [], []
            for f in frames:
                pts, vis = get_pose_points(f, landmarker)
                pts2d_list.append(pts)
                vis_list.append(vis)

            # triangulate
            targets_world = triangulate_landmarks(Ps, pts2d_list)
            common_ids = sorted(targets_world.keys())

            # need at least hips + shoulders for solve_pose
            required = {11, 12, 23, 24}
            if not required.issubset(set(common_ids)):
                print(f"  frame {frame_idx}: missing key landmarks, skipping")
                yield frame_idx, None
            else:
                # optional BA
                if run_ba and len(common_ids) >= 6:
                    _, _, Ps_ba, targets_world = bundle_adjustment(
                        K, Rs, ts, targets_world, pts2d_list, common_ids,
                        max_nfev=100,
                    )
                    # note: we don't update Rs/ts permanently — cameras are fixed

                pose_params = solve_pose(
                    targets_world, rest_out, rest_bone_poses, label_to_idx, model
                )
                yield frame_idx, pose_params

            # --- read next set of frames ---
            frames = []
            all_ok = True
            for cap in caps:
                ok, frame = cap.read()
                if not ok:
                    all_ok = False
                    break
                frames.append(cv2.resize(frame, size))

            if not all_ok:
                break

            frame_idx += 1

    finally:
        landmarker.close()
        for cap in caps:
            cap.release()


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("[setup] building Anny model ...")
    model = build_model()
    bone_labels = list(model.bone_labels)
    label_to_idx = {n: i for i, n in enumerate(bone_labels)}
    parent_indices = get_parent_indices(model)

    print("[setup] computing rest pose ...")
    rest_out, rest_bone_poses = run_rest_pose(model)

    K = get_camera_matrix(WIDTH, HEIGHT)

    print("[stream] starting ...")
    for frame_idx, pose_params in stream_pose_params(
        VIDEOS, K, model, rest_out, rest_bone_poses,
        label_to_idx, MEDIAPIPE_MODEL,
        size=(WIDTH, HEIGHT),
        run_ba=False,       # set True for accuracy, False for speed
    ):
        if pose_params is None:
            continue

        # ---- do whatever you want with pose_params here ----
        # e.g. render, send to viewer, save to disk, etc.
        print(f"  frame {frame_idx}: got {len(pose_params)} bone params")

In [ ]:
import trimesh

def main(pose_params):
    model = anny.create_fullbody_model(
        all_phenotypes=True,
        local_changes=True,
        remove_unattached_vertices=True,
        triangulate_faces=True,
    ).to(device=DEVICE, dtype=torch.float32)

    phenotype_kwargs = {
        "gender": 0,
        "age": 0.5,
        "height": 0.6,
        "weight": 1.0,
        "muscle": 0.8,
    }

    with torch.no_grad():
        output = model(
            pose_parameters=pose_params,
            phenotype_kwargs=phenotype_kwargs,
        )

    vertices = output["vertices"].squeeze(0).cpu().numpy()
    faces = model.faces.cpu().numpy()

    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    mesh.visual = trimesh.visual.TextureVisuals(
        material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[0.55, 0.78, 0.82, 1.0],
            metallicFactor=0.0,
            roughnessFactor=0.7,
            doubleSided=True,
        )
    )

    print(f"Vertices: {len(vertices)}")
    print(f"Faces: {len(faces)}")

    return mesh, vertices

mesh, vertices = main(pose_params)
mesh.show()